In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/dirty_cafe_sales.csv')
print("Shape:", df.shape)
df.head(10)

Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [3]:
print("Column names:")
print(df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)

Column names:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Data types:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


In [4]:
print("Null counts:")
print(df.isnull().sum())
print()
print("'ERROR' counts:")
print((df == 'ERROR').sum())
print()
print("'UNKNOWN' counts:")
print((df == 'UNKNOWN').sum())

Null counts:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

'ERROR' counts:
Transaction ID        0
Item                292
Quantity            170
Price Per Unit      190
Total Spent         164
Payment Method      306
Location            358
Transaction Date    142
dtype: int64

'UNKNOWN' counts:
Transaction ID        0
Item                344
Quantity            171
Price Per Unit      164
Total Spent         165
Payment Method      293
Location            338
Transaction Date    159
dtype: int64


In [5]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Duplicate Transaction IDs:", df['Transaction ID'].duplicated().sum())

Duplicate rows: 0

Duplicate Transaction IDs: 0


In [6]:
for col in ['Item', 'Payment Method', 'Location']:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Payment Method ---
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

--- Location ---
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64



In [7]:
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))
    print()

--- Quantity ---
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

--- Price Per Unit ---
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

--- Total Spent ---
Total Spent
6.0        979
12.0       939
3.0        930
4.0        923
20.0       746
15.0       734
8.0        677
10.0       524
2.0        497
9.0        479
5.0        468
16.0       444
25.0       259
7.5        237
1.0        232
4.5        225
1.5        205
NaN        173
UNKNOWN    165
ERROR      164
Name: count, dtype: int64



In [8]:
print("Sample date values:")
print(df['Transaction Date'].value_counts(dropna=False).head(20))
print()
print("Non-standard dates (not matching YYYY-MM-DD):")
mask = df['Transaction Date'].notna() & \
       ~df['Transaction Date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
print(df.loc[mask, 'Transaction Date'].value_counts(dropna=False))

Sample date values:
Transaction Date
UNKNOWN       159
NaN           159
ERROR         142
2023-06-16     40
2023-02-06     40
2023-09-21     39
2023-07-21     39
2023-07-24     39
2023-03-13     39
2023-01-05     38
2023-10-22     37
2023-06-18     37
2023-01-25     37
2023-04-30     36
2023-08-07     36
2023-11-07     36
2023-06-30     36
2023-01-12     36
2023-11-23     36
2023-10-08     35
Name: count, dtype: int64

Non-standard dates (not matching YYYY-MM-DD):
Transaction Date
UNKNOWN    159
ERROR      142
Name: count, dtype: int64


In [9]:
df_valid = df[
    ~df['Quantity'].isin(['ERROR', 'UNKNOWN']) & df['Quantity'].notna() &
    ~df['Price Per Unit'].isin(['ERROR', 'UNKNOWN']) & df['Price Per Unit'].notna() &
    ~df['Total Spent'].isin(['ERROR', 'UNKNOWN']) & df['Total Spent'].notna()
].copy()

df_valid['Quantity'] = pd.to_numeric(df_valid['Quantity'])
df_valid['Price Per Unit'] = pd.to_numeric(df_valid['Price Per Unit'])
df_valid['Total Spent'] = pd.to_numeric(df_valid['Total Spent'])

df_valid['expected_total'] = df_valid['Quantity'] * df_valid['Price Per Unit']
mismatch = df_valid[df_valid['Total Spent'] != df_valid['expected_total']]
print("Rows where Total Spent != Quantity * Price Per Unit:", len(mismatch))
print()
print(mismatch[['Quantity', 'Price Per Unit', 'Total Spent', 'expected_total']].head(10))

Rows where Total Spent != Quantity * Price Per Unit: 0

Empty DataFrame
Columns: [Quantity, Price Per Unit, Total Spent, expected_total]
Index: []


In [10]:
print("""
=== DATA EXPLORATION SUMMARY ===

Shape: 10,000 rows x 8 columns

DIRTY VALUES (NaN + ERROR + UNKNOWN):
- Item:              333 NaN, 292 ERROR, 344 UNKNOWN
- Quantity:          138 NaN, 170 ERROR, 171 UNKNOWN
- Price Per Unit:    179 NaN, 190 ERROR, 164 UNKNOWN
- Total Spent:       173 NaN, 164 ERROR, 165 UNKNOWN
- Payment Method:   2579 NaN, 306 ERROR, 293 UNKNOWN
- Location:         3265 NaN, 358 ERROR, 338 UNKNOWN
- Transaction Date:  159 NaN, 142 ERROR, 159 UNKNOWN

VALID CATEGORIES:
- Item:           Coffee, Tea, Cake, Cookie, Sandwich, Salad, Smoothie, Juice
- Payment Method: Cash, Credit Card, Digital Wallet
- Location:       In-store, Takeaway

DATE FORMAT: YYYY-MM-DD (consistent, no reformatting needed)
DUPLICATES: None
TOTAL SPENT: Always consistent with Quantity * Price Per Unit
""")


=== DATA EXPLORATION SUMMARY ===

Shape: 10,000 rows x 8 columns

DIRTY VALUES (NaN + ERROR + UNKNOWN):
- Item:              333 NaN, 292 ERROR, 344 UNKNOWN
- Quantity:          138 NaN, 170 ERROR, 171 UNKNOWN
- Price Per Unit:    179 NaN, 190 ERROR, 164 UNKNOWN
- Total Spent:       173 NaN, 164 ERROR, 165 UNKNOWN
- Payment Method:   2579 NaN, 306 ERROR, 293 UNKNOWN
- Location:         3265 NaN, 358 ERROR, 338 UNKNOWN
- Transaction Date:  159 NaN, 142 ERROR, 159 UNKNOWN

VALID CATEGORIES:
- Item:           Coffee, Tea, Cake, Cookie, Sandwich, Salad, Smoothie, Juice
- Payment Method: Cash, Credit Card, Digital Wallet
- Location:       In-store, Takeaway

DATE FORMAT: YYYY-MM-DD (consistent, no reformatting needed)
DUPLICATES: None
TOTAL SPENT: Always consistent with Quantity * Price Per Unit

